## Install necessary libraries

In [ ]:
!pip install qiskit qiskit_aer
!pip install qiskit_ibm_runtime

## Import the libraries

In [ ]:
import qiskit
print(qiskit.__version__)

In [ ]:
import math
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
import hmac
import hashlib

## Functions To generate shared bases
#### Based on the paper

In [ ]:
len_message = 16  # Length of binary output

def hmac_prf(secret_key: str, message: bytes) -> str:
    """
    Generate HMAC using SHA-256 and return a binary output truncated to `len_message` bits.

    Args:
    - secret_key (str): Shared secret key.
    - message (bytes): Input message (previous value + seed + counter).

    Returns:
    - str: Binary string representation (16 bits).
    """
    hmac_output = hmac.new(secret_key.encode(),message, hashlib.sha256).hexdigest()
    binary_output = bin(int(hmac_output, 16))[2:].zfill(256)[:len_message]
    return binary_output

def binary_concat(*args) -> bytes:
    """
    Concatenate multiple binary elements.

    Args:
    - *args: List of elements (str or bytes).

    Returns:
    - bytes: Concatenated result.
    """
    return b''.join(arg.encode() if isinstance(arg, str) else arg for arg in args)

def update_key_and_seed(K: str, S: str) -> tuple:
    """
    Generate a new K and S using HMAC-PRF to ensure forward secrecy.

    Args:
    - K (str): Previous secret key.
    - S (str): Previous seed.

    Returns:
    - (new_K, new_S): Tuple of updated K and S as hexadecimal strings.
    """
    new_K = hmac_prf(K, b"update_key")
    new_S = hmac_prf(S, b"update_seed")
    return new_K, new_S

def generate_next_T(K: str, S: str, prev_T: str, round_num: int) -> str:
    """
    Generate the next T value dynamically based on the previous T.

    Args:
    - K (str): Current secret key.
    - S (str): Current seed.
    - prev_T (str): Previous T value.
    - round_num (int): Round number for counter.

    Returns:
    - str: New T value in binary.
    """
    counter = round_num.to_bytes(1, byteorder='big')
    next_T = hmac_prf(K, \
                      binary_concat(prev_T.encode(),\
                      S.encode(), counter))
    return next_T

## Round 1

In [ ]:
# === Initialization ===
K = "shared_secret_key"
S = "shared_seed"
prev_T = "0000000000000000"

# === Run a single round and show changes ===
round_number = 1
print(f"Round {round_number}:")
print(f"  Initial K  = {K}")
print(f"  Initial S  = {S}")

# Generate new T
new_T = generate_next_T(K, S, prev_T, round_number)
print(f"  Generated T{round_number} = {new_T}")

# Update K and S for the next round
K, S = update_key_and_seed(K, S)
print(f"  Updated K  = {K}")
print(f"  Updated S  = {S}")

In [ ]:
round_number = 2
print(f"Round {round_number}:")
print(f"  Initial K  = {K}")
print(f"  Initial S  = {S}")


new_T = generate_next_T(K, S, prev_T, round_number)
print(f"  Generated T{round_number} = {new_T}")

# Update K and S for the next round
K, S = update_key_and_seed(K, S)
print(f"  Updated K  = {K}")
print(f"  Updated S  = {S}")


## Select common bases

In [ ]:
alice_bases = bob_bases = new_T

## Runtime initialisation

In [ ]:
import os
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

QiskitRuntimeService.save_account(
  token=os.environ["IBM_QUANTUM_TOKEN"],
  channel="ibm_cloud",
  overwrite=True
)


In [ ]:
service = QiskitRuntimeService()
n_qubits = 5
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=n_qubits)
print("Selected Backend", backend)

In [ ]:
print(backend)

## Quantum Random Number Generation

In [ ]:
# Function to generate a random binary string using quantum circuits
def RandomStringIBM(str_len):
    op_str = ''  # Initialize an empty output string
    num_qbits = 16  # Define number of qubits

    # Calculate the number of chunks needed
    num_chunks = math.ceil(str_len / num_qbits)
    for _ in range(num_chunks):
        # Create a quantum register and a classical register
        q = QuantumRegister(num_qbits)
        c = ClassicalRegister(num_qbits)
        QC = QuantumCircuit(q, c)

        # Apply Hadamard gates to all qubits to create superposition
        for i in range(num_qbits):
            QC.h(q[i])
        QC.measure(q, c)
        print(QC.draw())
        pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
        isa_circuit = pm.run(QC)

        sampler = Sampler(backend)
        job = sampler.run([isa_circuit])
        print(job.job_id)
        result = job.result()


In [ ]:
len_message = 16
alice_key = RandomStringIBM(len_message)

### Job ID

Random Number : cyctrvvcw2k0008kns7g


In [ ]:
job_id = 'd7ah150eecps73d9he60'
job = service.job(job_id)
result = job.result()
data = result[0].data
bitarray = next(iter(data.values()))
counts = bitarray.get_counts()
alice_key = next(iter(counts.keys()))


In [ ]:
print(alice_key)

In [ ]:
# Quantum Circuit for encoding
q = QuantumRegister(len_message)
c = ClassicalRegister(len_message)
qc = QuantumCircuit(q, c)

# Encode qubits based on Alice's key and basis
for i in range(len_message):
    if alice_key[i] == '1':
        qc.x(q[i])  # Flip the qubit to 1 if key is 1
    if alice_bases[i] == '1':
        qc.h(q[i])  # Apply Hadamard gate if basis is 1
qc.barrier()

In [ ]:
# Step 3: Bob measures the qubits
for i in range(len_message):
    if bob_bases[i] == '1':
        qc.h(q[i])  # Bob applies Hadamard if measuring in Hadamard basis

# Add measurements
qc.measure(q, c)

In [ ]:
qc.draw()

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = Sampler(backend)
job = sampler.run([isa_circuit],shots=1)
result = job.result()


In [ ]:
data = result[0].data
bitarray = next(iter(data.values()))
counts = bitarray.get_counts()

## Run the circuit

Circuit : cyctsyf01rbg008k06yg

In [ ]:

job_id = 'cyctsyf01rbg008k06yg'
job = service.job(job_id)
result = job.result()
data = result[0].data
bitarray = next(iter(data.values()))
counts = bitarray.get_counts()
bob_result = next(iter(counts.keys()))


In [ ]:
bob_result = next(iter(counts.keys()))
print(bob_result)

In [ ]:
print((alice_bases))
print((bob_bases))
print((alice_key))
print((bob_result))


In [ ]:
# Display matching bits for shared key
shared_key = []
for i in range(len_message):
    if alice_bases[i] == bob_bases[i]:  # Only keep the bit if bases match
        shared_key.append(bob_result[len_message - i - 1])  # Reverse order in Qiskit

final_shared_key = ''.join(shared_key)
print('Final Shared Key:', final_shared_key)

# Calculate the percentage of key bits retained
matching_bits_count = len(final_shared_key)
percentage_retained = (matching_bits_count / len_message) * 100
print("Percentage of key bits retained: ", percentage_retained, "%")

## One time pad

In [ ]:
final_shared_key = 1010001100011111

fin

In [ ]:
import random

# Function to convert a string to binary
def text_to_bin(text):
    return ''.join(format(ord(c), '08b') for c in text)

# Function to convert binary back to string
def bin_to_text(binary_string):
    chars = [binary_string[i:i+8] for i in range(0, len(binary_string), 8)]
    return ''.join(chr(int(char, 2)) for char in chars)

# Function to encrypt a message using One-Time Pad
def encrypt(message, key):
    # Convert the message to binary
    binary_message = text_to_bin(message)

    # Encrypt by XORing the binary message with the key (key length fixed at 28)
    encrypted = ''.join(str(int(binary_message[i]) ^ int(key[i % len(key)])) for i in range(len(binary_message)))
    return encrypted

# Function to decrypt a message using One-Time Pad
def decrypt(ciphertext, key):
    # Decrypt by XORing the ciphertext with the key
    decrypted = ''.join(str(int(ciphertext[i]) ^ int(key[i % len(key)])) for i in range(len(ciphertext)))

    # Convert the decrypted binary string back to text
    return bin_to_text(decrypted)

# Example usage
message = "Helod World"

# Assuming final_shared_key is predefined and has a length of 28

print(f"Key: {final_shared_key}")

# Encrypt the message using One-Time Pad
ciphertext = encrypt(message, final_shared_key)
print(f"Encrypted Message (Binary): {ciphertext}")

# Decrypt the message using the same key
decrypted_message = decrypt(ciphertext, final_shared_key)
print(f"Decrypted Message: {decrypted_message}")


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:us-east:a/25040d1a4afb4abf8b6601759a19f289:782d5680-fbdf-470b-8e6b-5201f9b8abe9::'
)
job = service.job('d7agio2k86tc73a1jk6g')
job_result = job.result()